# Instationäre Behälterleerung -- zweiter Lösungsweg: rein numerisch

**BIW3-10 Weiterführende Hydromechanik · Übung 2 · Aufgabe 5**

Auf dem Blatt wurde die Bewegungsgleichung hergeleitet (Volumenbilanz $+$ Torricelli) und $h(t)$ geschlossen gelöst. **Dieses Notebook geht einen zweiten Weg und braucht die hergeleitete Gleichung dafür nicht:**

- die **Physik wird direkt programmiert**: der momentane Abfluss nach Torricelli senkt den Spiegel (Kontinuität),
- **scipy** integriert die Bewegung in der Zeit,
- die hergeleitete Gleichung $h(t) = h_0\left(1-\frac{t}{T_E}\right)^2$ taucht **erst ganz am Ende** auf -- als Kontrolle der Numerik.

**Lernziele:** Physik als Funktion `f(t, y)` aufschreiben · mit `solve_ivp` lösen · die Instationarität in der Zeitreihe sehen · begründen, warum ohne Herleitung dieselbe Kurve entsteht.

## 1. Parameter und Flächen

Erste Programmierübung: tragen Sie die Werte aus Aufgabe 4 des Übungsblatts ein (**SI-Einheiten**, also Meter und Sekunden!) und berechnen Sie die beiden Querschnitte als Kreisflächen.

In [ ]:
# ======================================================================
# 1. PARAMETER UND FLÄCHEN  (Werte aus Aufgabe 4, SI-Einheiten)
# ======================================================================
g     = 9.81      # m/s^2   Erdbeschleunigung
D     = 2.0       # m       Behälter-Innendurchmesser
d_o   = 0.08      # m       Öffnungsdurchmesser (Blatt: 80 mm)
alpha = 0.62      # -       Ausflusszahl
h0    = 3.6       # m       Anfangswasserstand

import numpy as np
import matplotlib.pyplot as plt

# Querschnitte als Kreisflächen
A   = np.pi * D**2 / 4    # m^2
A_o = np.pi * d_o**2 / 4  # m^2
# Kontrolle:
print(f"A   = {A:.6f} m²")
print(f"A_o = {A_o:.6f} m²")

## 2. Die Physik als Funktion (Aufgabe 5a)

**Ohne jede hergeleitete Formel** -- nur zwei Sätze aus der Vorlesung:

1. **Torricelli** (Folie 16): der Abfluss hängt vom *momentanen* Wasserstand ab,
   $$Q_{\mathrm{ab}} = \alpha\, A_{\mathrm{o}}\,\sqrt{2 g h}$$
2. **Kontinuität** (Volumenbilanz): der Abfluss senkt den Spiegel um $Q_{\mathrm{ab}}/A$ pro Zeit,
   $$\frac{\mathrm{d}h}{\mathrm{d}t} = -\frac{Q_{\mathrm{ab}}}{A}$$

Diese zwei Zeilen sind die **rechte Seite** $f(t,h)$ -- mehr braucht `scipy` nicht. Jede Auswertung ist eine Momentaufnahme (quasi-stationär, Folie 26).

In [ ]:
# ======================================================================
# 3. DIE PHYSIK ALS FUNKTION   (Aufgabe 5a -- LÖSUNG)
# ======================================================================
def rhs(t, y):
    """Rechte Seite der Bewegung für solve_ivp.
        t : Zeit [s] (unbenutzt),  y : Zustandsvektor, y[0] = h [m].
        Rückgabe: dh/dt [m/s].
    """
    h = y[0]

    # TODO 1 (gelöst): Torricelli mit dem momentanen Wasserstand
    Q_ab = alpha * A_o * np.sqrt(2 * g * max(h, 0.0))     # m^3/s

    # TODO 2 (gelöst): Kontinuität -- Q_ab senkt den Spiegel um Q_ab/A
    dhdt = -Q_ab / A                                      # m/s

    return dhdt

# Kontrolle:
print(f"Q_ab(0) = {alpha*A_o*np.sqrt(2*g*h0)*1000:.1f} L/s   (soll 26,2 L/s)")
# Mini-Test: Anfangssteigung (soll -8,34 mm/s sein -- maximal, weil viel Wasser drückt)
print(f"dh/dt(0) = {rhs(0, [h0])*1000:.2f} mm/s")

## 3. Zeitintegration mit scipy (Aufgabe 5b)

`scipy.integrate.solve_ivp` integriert $\dd y/\dd t = f(t,y)$ automatisch (Runge-Kutta 4/5, **adaptive Schrittweite** -- groß, wo sich wenig tut; klein, wo es steil wird). Wir übergeben:

- die **Physik-Funktion** `rhs` (von oben),
- den **Zeitbereich** `(0, 3600)` -- eine ganze Stunde, großzügig gewählt: ohne die hergeleitete Formel kennen wir die Auslaufzeit noch nicht, das Event stoppt die Rechnung automatisch,
- den **Anfangszustand** `[h0]` (Vektor! später wären mehrere Unbekannte möglich),
- ein **Ereignis (Event)**: "Behälter leer" bei $h = 0$ -- dort stoppt die Rechnung,
- `dense_output=True`: liefert zusätzlich eine stetige Lösungsfunktion `sol.sol(t)` (glatte Plots, Sensor-Zeitpunkt),
- Genauigkeit: `rtol=1e-8`, `atol=1e-10`.

In [ ]:
# ======================================================================
# 4. ZEITINTEGRATION   (Aufgabe 5b -- LÖSUNG)
# ======================================================================
from scipy.integrate import solve_ivp

def event_leer(t, y):
    return y[0]                     # TODO 3 (gelöst): Event bei h = 0

event_leer.terminal = True          # bei h = 0 anhalten
event_leer.direction = -1           # nur + -> - Übergänge zählen

t_span = (0, 3600)               # 1 Stunde -- großzügig, das Event stoppt rechtzeitig
y0 = [h0]                           # Anfangsbedingung als VEKTOR
solver = dict(rtol=1e-8, atol=1e-10)

# TODO 4 (gelöst): Zeitintegration
sol = solve_ivp(rhs, t_span, y0, events=event_leer, dense_output=True, **solver)

# TODO 5 (gelöst): Auslaufzeit aus dem Event
T_E_num = sol.t_events[0][0]

print(f"T_E,num = {T_E_num:.2f} s = {T_E_num/60:.2f} min")

# Instationarität sichtbar: die Absenkgeschwindigkeit nimmt ab
print(f"dh/dt(0)      = {rhs(0, [h0])*1000:+.2f} mm/s  (maximal)")
print(f"dh/dt(0.5T_E) = {rhs(T_E_num/2, [h0/4])*1000:+.2f} mm/s  (halber Abfluss, da Q ~ sqrt(h))")

# Kontrolle:
print(f"Endwasserstand = {sol.y[0][-1]:.2e} m   (soll 0)")
print(f"Zeitschritte  = {len(sol.t)}   (adaptiv)")

## 4. Ergebnis: die Zeitreihe (Aufgabe 5c)

Die gelöste Zeitreihe $h(t)$ ist die Instationarität zum Anfassen: am festen Ort (dem Behälter) ändert sich der Zustand ständig -- $\partial h/\partial t \neq 0$. Der Abfluss fällt mit: $Q(t) = \alpha A_{\mathrm{o}}\sqrt{2g\,h(t)}$.

In [ ]:
# ======================================================================
# 5. PLOT   (Aufgabe 5c -- LÖSUNG)
# ======================================================================
t_fein = np.linspace(0, sol.t[-1], 500)
h_fein = sol.sol(t_fein)[0]

plt.figure(figsize=(8, 4))
plt.plot(t_fein, h_fein, 'b-', linewidth=2, label='numerisch: h(t)')

# TODO 6 (gelöst): Sensor-Wasserstand einzeichnen
plt.axhline(1.0, color='orange', linestyle='--', label='Sensor (1,0 m)')

plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Instationäre Behälterleerung -- rein numerisch gerechnet')
plt.show()

# Kontrolle: Sensor-Zeitpunkt (Aufgabe 4e)
t_sensor = t_fein[np.argmax(h_fein < 1.0)]
print(f"Sensor erreicht bei t = {t_sensor:.1f} s   (Aufgabe 4e: ~408 s)")

# Ablesbar: der Sensor bei h = 1,0 m wird bei t ~ 408 s (6,8 min) erreicht --
# das ist genau der Wert aus Aufgabe 4e (ohne dass die Formel im Code steckte).

## 5. Erst jetzt: Vergleich mit der hergeleiteten Gleichung

**Jetzt** holen wir die aus der Übung hergeleitete Gleichung -- sie dient nur noch als **Kontrolle** der Numerik:

$$h(t) = h_0\left(1-\frac{t}{T_E}\right)^{2}, \qquad T_E = \frac{2A\sqrt{h_0}}{\alpha A_{\mathrm{o}}\sqrt{2g}}$$

Wenn der numerische Weg korrekt ist, muss er auf derselben Kurve landen -- **ohne dass die Formel irgendwo in den Code eingegangen ist**.

In [ ]:
# ======================================================================
# 6. VERGLEICH MIT DER HERGELEITETEN GLEICHUNG   (Kontrolle)
# ======================================================================
# Die Formel aus der Übung -- erst HIER kommt sie vor:
T_E_exakt = 2 * A * np.sqrt(h0) / (alpha * A_o * np.sqrt(2*g))   # s

def h_hergeleitet(t):
    """Aus der Übung: h(t) = h0*(1 - t/T_E)^2 -- Kontrollkurve."""
    return h0 * (1 - t/T_E_exakt)**2

t_fein = np.linspace(0, T_E_exakt, 200)

plt.figure(figsize=(8, 4))
plt.plot(t_fein, h_hergeleitet(t_fein), 'r--', linewidth=1.5, label='hergeleitete Gleichung (exakt)')
plt.plot(sol.t, sol.y[0], 'b-', linewidth=2, label='numerisch (ohne Herleitung)')
plt.axhline(1.0, color='orange', linestyle=':', label='Sensor (1,0 m)')
plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Zweiter Lösungsweg trifft den ersten')
plt.show()

print(f"T_E,num   = {T_E_num:.2f} s")
print(f"T_E,exakt = {T_E_exakt:.2f} s")
print(f"Abweichung = {(T_E_num - T_E_exakt)/T_E_exakt*100:+.4f} %")

# Zusatz: Anfangs- und Endverhalten der Absenkung (Instationarität!)
print(f"dh/dt(0)      = {rhs(0, [h0])*1000:+.2f} mm/s  (maximale Absenkung)")
print(f"dh/dt(0.5T_E) = {rhs(T_E_exakt/2, [h0/4])*1000:+.2f} mm/s  (halber Abfluss, da Q ~ sqrt(h))")

## 6. Interpretation (Aufgabe 5d) – Erwartete Antworten

1. **Dieselbe Parabel ohne Herleitung**, weil beide Wege *dieselbe Physik kodieren*: `rhs` enthält Torricelli ($Q \propto \sqrt{h}$) und die Volumenbilanz ($\div A$) -- exakt die zwei Schritte der Blatt-Herleitung. Die geschlossene Form ist nur die mathematische Abkürzung dafür.
2. **Instationär, weil $\partial h/\partial t \neq 0$:** am festen Ort (Behälter) ändert sich der Zustand ständig -- $h(t)$ fällt, $Q(t)$ fällt von 26,2 L/s auf 0. Der Zeitableitungs-Term $A\,\dd h/\dd t$ ist der neue Term gegenüber der stationären Bilanz.
3. **Am Anfang am schnellsten**, weil $Q_{\mathrm{ab}} \propto \sqrt{h}$: bei $h_0 = 3{,}6$ m fließen 26,2 L/s ($\dd h/\dd t = -8{,}3$ mm/s), bei $h_0/4$ nur noch die Hälfte -- die Kurve wird flach.
4. **Linearer Spiegelfall**, wenn `rhs` einen konstanten Abfluss kodiert: $\dd h/\dd t = -Q_{\mathrm{p}}/A$ (Pumpe, Aufgabe 3a) -- die Kurvenform steckt vollständig in der programmierten Physik.